<a href="https://colab.research.google.com/github/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/blob/main/hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import itertools
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score

URL = "https://raw.githubusercontent.com/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/main/Mall_Customers.csv"
FEATURES = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

df = pd.read_csv(URL).drop_duplicates().dropna(subset=FEATURES)
X = StandardScaler().fit_transform(df[FEATURES])

param_grid = {
    "n_clusters": [2, 3, 4, 5],
    "init": ["k-means++", "random"],
    "n_init": [5, 10],
    "max_iter": [100, 300],
}

def tune_kmeans(X, grid):
    results = []
    keys = list(grid.keys())
    for values in itertools.product(*grid.values()):
        params = dict(zip(keys, values))
        model = KMeans(random_state=42, **params)
        labels = model.fit_predict(X)
        if len(set(labels)) < 2:
            continue
        results.append({
            **params,
            "silhouette": round(silhouette_score(X, labels), 3),
            "calinski_harabasz": round(calinski_harabasz_score(X, labels), 1),
            "inertia": round(model.inertia_, 2),
        })
    return pd.DataFrame(results).sort_values("silhouette", ascending=False).reset_index(drop=True)

results = tune_kmeans(X, param_grid)
print("Total combinations tried:", len(results))
print("\nTop 5 combinations:")
print(results.head(5))

best = results.iloc[0]
print("\nBest parameters:")
print(best[["n_clusters", "init", "n_init", "max_iter"]].to_string())
print("Best silhouette score:", best["silhouette"])


Total combinations tried: 32

Top 5 combinations:
   n_clusters       init  n_init  max_iter  silhouette  calinski_harabasz  \
0           2  k-means++       5       100       0.394               15.1   
1           2  k-means++       5       300       0.394               15.1   
2           2  k-means++      10       100       0.394               15.1   
3           2  k-means++      10       300       0.394               15.1   
4           2     random       5       100       0.394               15.1   

   inertia  
0     32.6  
1     32.6  
2     32.6  
3     32.6  
4     32.6  

Best parameters:
n_clusters            2
init          k-means++
n_init                5
max_iter            100
Best silhouette score: 0.394
